In [6]:
import os 
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import ollama

In [5]:
MODEL = "qwen3:8b"

In [10]:
system_message = """You are a helpful assistant for an airline called FlightAI. 
Give short, courteous answers, no more than 1 sentence. 
Always be correct. If you don't know the answer, say so."""

In [11]:
def chat(message,history):
     history = [{"role":h["role"],"content":h["content"]} for h in history]
     messages = [{"role":"system","content":system_message}] + history + [{"role":"user","content":message}]
     response = ollama.chat(
          model=MODEL,
          messages=messages,
     )
     return response["message"]["content"]


In [12]:
gr.ChatInterface(
    fn=chat
).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print (f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower())
    return f"The price of a ticket to {destination_city} is {price}"

In [24]:

price_function = {
    "type": "function",
    "function": {
        "name": "get_ticket_price",
        "description": "Get the price of a return ticket to the destination city.",
        "parameters": {
            "type": "object",
            "properties": {
                "destination_city": {
                    "type": "string",
                    "description": "The city that the customer wants to travel to"
                }
            },
            "required": [
                "destination_city"
            ]
        }
    }
}

In [25]:


tools = [
    price_function
]

In [26]:
# Cell 11 — Handle Tool Calls

def handle_tool_calls(message):

    responses = []

    for tool_call in message.tool_calls:

        if tool_call.function.name == "get_ticket_price":

            arguments = tool_call.function.arguments

            city = arguments.get("destination_city")

            price_details = get_ticket_price(city)

            responses.append({
                "role": "tool",
                "content": price_details
            })

    return responses

In [ ]:

def chat(message, history):

    messages: list[dict] = [
        {
            "role": "system",
            "content": system_message
        }
    ]

    for h in history:

        content = h["content"]

        if isinstance(content, list):
            content = "".join(
                item["text"]
                for item in content
                if isinstance(item, dict) and "text" in item
            )

        messages.append({
            "role": h["role"],
            "content": str(content or "")
        })

    messages.append({
        "role": "user",
        "content": message
    })

    response = ollama.chat(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    if response.message.tool_calls:

        message_with_tools = response.message

        responses = handle_tool_calls(
            message_with_tools
        )

        messages.append({
            "role": "assistant",
            "content": message_with_tools.content or "",
            "tool_calls": [
                {
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                }
                for tool_call in (message_with_tools.tool_calls or [])
            ]
        })

        messages.extend(responses)

        response = ollama.chat(
            model=MODEL,
            messages=messages
        )

    return response.message.content

In [30]:
gr.ChatInterface(
    fn= chat,
).launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
